In [1]:
import numpy as np
import pandas as pd

from statsmodels.formula.api import logit, probit, ols

# Do not show FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

from IPython.display import Markdown, display

# load datasets as a dictionary
path_to_datasets = '../../datasets/econometrica-2/'
df_dict = {'loanapp': pd.read_csv(path_to_datasets+'loanapp.csv'), 
		'TableF5-1':pd.read_csv(path_to_datasets+'TableF5-1.csv'), 
		'SwissLabour': pd.read_csv(path_to_datasets+'SwissLabour.csv')}

## Marginal values/effects

### Formulas for marginal effects

__for the probit model__ $P(y=1)=\Phi(x^\top\beta)$ the formula for marginal values is

$$
\begin{aligned}
	\frac{\partial P(y=1)}{\partial x_j}&=\phi\left(x^\top\beta\right)\beta_j & \phi(z)=\frac{1}{\sqrt{2\pi}}exp\left(-z^2/2\right)
\end{aligned}
$$

__for the logit model__ $P(y=1)=\Lambda(x^\top\beta)$ the formula for marginal values is

$$
\begin{aligned}
	\frac{\partial P(y=1)}{\partial x_j}&=\lambda\left(x^\top\beta\right)\beta_j & \lambda(z)=\frac{exp(z)}{(1+exp(z))^2}
\end{aligned}
$$

### Average marginal effects

Typically the following are calculated

- marginal values for each regressor at the mean point $x_{mean}=\sum_i x_i/n$:

$$
\begin{aligned}
	probit:&\ \phi\left(x_{mean}^\top\beta\right)\beta_j & logit:&\ \lambda\left(x_{mean}^\top\beta\right)\beta_j
\end{aligned}
$$

- the sample-average marginal value for each regressor:

$$
\begin{aligned}
	probit:&\ \frac 1n \sum_{i}\phi\left(x_i^\top\beta\right)\beta_j & logit:&\ \frac 1n \sum_{i}\lambda\left(x_i^\top\beta\right)\beta_j
\end{aligned}
$$

In [2]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =['I(appinc/100)+mortno+unem+dep+male',
			'WA+np.log(FAMINC)+WE+KL6+K618+CIT+UN',
			'income+age+I(age**2)+youngkids+oldkids']

mod_name = ['Approve equation', 'labour force equation', 'swiss labour force equation']

sign_level = [0.05, 0.01, 0.10]

digits = 3

for i, df_name in enumerate(df_names):
	mod_logit  = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_probit = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_ols = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res_logit = mod_logit.fit(disp=False)
	res_probit = mod_probit.fit(disp=False)
	res_ols = mod_ols.fit(cov_type='HC3')

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'For the dataset `{df_name}`, estimate the __probit__ and __logit__ regressions of `{endog_var[i]}` on `{exog_vars[i]}`. For each regression, compute the marginal values and interpret them:\n'
	f'- at the mean point for each regressor \n'
	f'- the sample-average marginal value for each regressor.\n\n '
	f'Test the significance of the marginal effects at the significance level {sign_level[i]*100}%. Compare with the marginal effects in the LPM model. __Round your answer to {digits} decimal places__'))

	df_margeff = pd.DataFrame({'Regressor': res_logit.params.index[1:], 
								'LPM effect': res_ols.params[1:].round(digits),
								'At mean point (logit)': res_logit.get_margeff(at='mean', dummy=True).margeff.round(digits),
								'At mean point (probit)': res_probit.get_margeff(at='mean', dummy=True).margeff.round(digits),
								'Sample average (logit)': res_logit.get_margeff(at='overall', dummy=True).margeff.round(digits),
								'Sample average (probit)': res_probit.get_margeff(at='overall', dummy=True).margeff.round(digits)
								})

	df_margeff_stats = pd.DataFrame({'Regressor': res_logit.params.index[1:], 
									'LPM effect': res_ols.tvalues[1:].round(digits),
									'At mean point (logit)': res_logit.get_margeff(at='mean', dummy=True).tvalues.round(digits),
									'At mean point (probit)': res_probit.get_margeff(at='mean', dummy=True).tvalues.round(digits),
									'Sample average (logit)': res_logit.get_margeff(at='overall', dummy=True).tvalues.round(digits),
									'Sample average (probit)': res_probit.get_margeff(at='overall', dummy=True).tvalues.round(digits)
									})

	df_margeff_sign = pd.DataFrame({'Regressor':res_logit.params.index[1:], 
								'LPM effect': res_ols.pvalues[1:].apply(lambda x: 'Significant' if x<sign_level[i] else 'Not significant'),
								'At mean point (logit)': res_logit.get_margeff(at='mean', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Significant' if x<sign_level[i] else 'Not significant'),
								'At mean point (probit)': res_probit.get_margeff(at='mean', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Significant' if x<sign_level[i] else 'Not significant'),
								'Sample average (logit)': res_logit.get_margeff(at='overall', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Significant' if x<sign_level[i] else 'Not significant'),
								'Sample average (probit)': res_probit.get_margeff(at='overall', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Significant' if x<sign_level[i] else 'Not significant')
								})

	display(Markdown(f'''__Answer__: marginal effects\n'''))
	display(Markdown(df_margeff.to_markdown(index=False)))
	display(Markdown(f'''their test statistics\n'''))
	display(Markdown(df_margeff_stats.to_markdown(index=False)))
	display(Markdown(f'''significance of marginal effects\n'''))
	display(Markdown(df_margeff_sign.to_markdown(index=False)))



### Approve equation

For the dataset `loanapp`, estimate the __probit__ and __logit__ regressions of `approve` on `I(appinc/100)+mortno+unem+dep+male`. For each regression, compute the marginal values and interpret them:
- at the mean point for each regressor 
- the sample-average marginal value for each regressor.

 Test the significance of the marginal effects at the significance level 5.0%. Compare with the marginal effects in the LPM model. __Round your answer to 3 decimal places__

__Answer__: marginal effects


| Regressor       |   LPM effect |   At mean point (logit) |   At mean point (probit) |   Sample average (logit) |   Sample average (probit) |
|:----------------|-------------:|------------------------:|-------------------------:|-------------------------:|--------------------------:|
| I(appinc / 100) |       -0.014 |                  -0.011 |                   -0.011 |                   -0.011 |                    -0.011 |
| mortno          |        0.079 |                   0.077 |                    0.077 |                    0.078 |                     0.077 |
| unem            |       -0.008 |                  -0.007 |                   -0.007 |                   -0.007 |                    -0.007 |
| dep             |       -0.012 |                  -0.011 |                   -0.011 |                   -0.011 |                    -0.011 |
| male            |        0.019 |                   0.019 |                    0.019 |                    0.019 |                     0.019 |

their test statistics


...

| Regressor   |   LPM effect |   At mean point (logit) |   At mean point (probit) |   Sample average (logit) |   Sample average (probit) |
|:------------|-------------:|------------------------:|-------------------------:|-------------------------:|--------------------------:|
| income      |       -0.258 |                  -0.33  |                   -0.318 |                   -0.279 |                    -0.278 |
| age         |        0.795 |                   0.965 |                    0.934 |                    0.817 |                     0.815 |
| I(age ** 2) |       -0.112 |                  -0.136 |                   -0.131 |                   -0.115 |                    -0.115 |
| youngkids   |       -0.228 |                  -0.264 |                   -0.253 |                   -0.224 |                    -0.221 |
| oldkids     |       -0.053 |                  -0.062 |                   -0.06  |                   -0.053 |                    -0.053 |

their test statistics


| Regressor   |   LPM effect |   At mean point (logit) |   At mean point (probit) |   Sample average (logit) |   Sample average (probit) |
|:------------|-------------:|------------------------:|-------------------------:|-------------------------:|--------------------------:|
| income      |       -7.533 |                  -6.249 |                   -6.428 |                   -6.788 |                    -6.898 |
| age         |        6.405 |                   5.849 |                    5.95  |                    6.288 |                     6.323 |
| I(age ** 2) |       -7.527 |                  -6.633 |                   -6.79  |                   -7.295 |                    -7.359 |
| youngkids   |       -7.029 |                  -6.458 |                   -6.69  |                   -7.082 |                    -7.232 |
| oldkids     |       -2.965 |                  -3.052 |                   -3.055 |                   -3.112 |                    -3.104 |

significance of marginal effects


| Regressor   | LPM effect   | At mean point (logit)   | At mean point (probit)   | Sample average (logit)   | Sample average (probit)   |
|:------------|:-------------|:------------------------|:-------------------------|:-------------------------|:--------------------------|
| income      | Significant  | Significant             | Significant              | Significant              | Significant               |
| age         | Significant  | Significant             | Significant              | Significant              | Significant               |
| I(age ** 2) | Significant  | Significant             | Significant              | Significant              | Significant               |
| youngkids   | Significant  | Significant             | Significant              | Significant              | Significant               |
| oldkids     | Significant  | Significant             | Significant              | Significant              | Significant               |